In [ ]:
import pandas as pd
import scipy.stats as stats
import altair as alt
import numpy as np
from pathlib import Path
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score

alt.data_transformers.disable_max_rows()

figure_date_of_record = '20260505'

# Data Processing Helper Functions

In [ ]:
def spliceai_mapper(df):

    df['maxSpliceAI'] = df[['spliceAI_DS_AG', 'spliceAI_DS_AL', 'spliceAI_DS_DG', 'spliceAI_DS_DL']].max(axis = 1)

    df['splice_impact'] = '< Threshold'
    SPLICE_IMPACT_COLS = {
        'spliceAI_DS_AG': 'Acceptor Gain',
        'spliceAI_DS_AL': 'Acceptor Loss',
        'spliceAI_DS_DG': 'Donor Gain',
        'spliceAI_DS_DL': 'Donor Loss',
    }

    for col, label in SPLICE_IMPACT_COLS.items():
        df.loc[df[col] >= 0.2, 'splice_impact'] = label
    
    df['simple_splice_impact'] = '< Threshold'
    
    df.loc[df['splice_impact'] != '< Threshold', 'simple_splice_impact'] = 'Splice Impact Predicted'
    return df

In [ ]:
def molecular_consequence_mapper(df, remap_col):

    df = df.dropna(subset=[remap_col]).copy()
    CONSEQUENCE_EXACT = {
            'synonymous_variant': 'Synonymous',
            'intron_variant':     'Intron',
            'stop_gained':        'Stop Gained',
            'stop_lost':          'Stop Lost',
            'start_lost':         'Start Lost',
            'inframe_indel':      'Inframe Indel',
        }

    CONSEQUENCE_CONTAINS = {
        'missense': 'Missense',
        'site':     'Canonical Splice',
        'ing_var':  'Splice Region',
        'UTR':      'UTR Variant',
    }

    df[remap_col] = df[remap_col].replace(CONSEQUENCE_EXACT)
    for pattern, label in CONSEQUENCE_CONTAINS.items():
        df.loc[df[remap_col].str.contains(pattern), remap_col] = label
    

    return df

In [ ]:
raw_df = pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna/external_rna_data/20260424_CAVASGE_SangerSGE_FindlaySGE.xlsx')

raw_df = raw_df.loc[(raw_df['ref_allele'].str.len()==1) & (raw_df['alt_allele'].str.len()==1)].copy()

raw_df['pos_id']=raw_df['Gene']+':'+raw_df['hg38_start'].astype(str)+':'+raw_df['alt_allele']
raw_df['auth_reported_func_class'] = raw_df['auth_reported_func_class'].replace({
    'LOF':           'functionally_abnormal',
    'LOF1':          'functionally_abnormal',
    'LOF2':          'functionally_abnormal',
    'depleted':      'functionally_abnormal',
    'slow depleted': 'functionally_abnormal',
    'fast depleted': 'functionally_abnormal',
    'slow depleting':'functionally_abnormal',
    'fast depleting':'functionally_abnormal',
    'enriched':      'functionally_normal',
    'FUNC':          'functionally_normal',
    'Neutral':       'functionally_normal',
    'unchanged':     'functionally_normal',
    'INT':           'indeterminate',
    'Intermediate':  'indeterminate',
})

df = spliceai_mapper(raw_df)
df = molecular_consequence_mapper(df, 'simplified_consequence')

cleaned_raw_df = df
df.head()


In [ ]:
def rna_merge_prepper(df, gene=None, pos_col='pos', alt_col='alt', threshold=None):

    if gene is not None:
        df['pos_id'] = gene + ':' + df[pos_col].astype(str) + ':' + df[alt_col]
    else:
        df['pos_id'] = df['Gene'] + ':' + df[pos_col].astype(str) + ':' + df[alt_col]
    df = df.dropna(subset=['rna_score']).copy()

    df['rna_consequence'] = 'normal'
    df.loc[df['rna_score'] <= threshold, 'rna_consequence'] = 'low'

    keep_cols = ['pos_id', 'rna_score', 'rna_consequence']
    if 'maxSpliceAI' in df.columns:
        keep_cols.append('maxSpliceAI')

    return df[keep_cols]

In [ ]:
vhl_data = pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna/external_rna_data/2025_VHLBuckley.xlsx')
vhl_data = vhl_data.replace({'LOF1': 'LoF', 'LOF2': 'LoF',
                             'STOP_GAINED': 'Nonsense',
                             'NON_SYNONYMOUS': 'Missense',
                             'SYNONYMOUS': 'Synonymous',
                             'CANONICAL_SPLICE': 'Canonical Splice',
                             'INTRONIC': 'Intronic',
                             'SPLICE_SITE': 'Splice Region',
                             'STOP_LOST': 'Stop Lost'})
vhl_data = vhl_data.rename(columns={'max_spliceAI': 'maxSpliceAI'})

# Snapshot SpliceAI before rna_merge_prepper drops rna_score-NaN rows.
# VHL intron variants have maxSpliceAI populated but no rna_score, so they
# would otherwise be dropped before reaching the merge in the next cell.
vhl_data['pos_id'] = 'VHL:' + vhl_data['hg38_pos'].astype(str) + ':' + vhl_data['alt']
vhl_spliceai = vhl_data[['pos_id', 'maxSpliceAI']].dropna(subset=['maxSpliceAI']).copy()

vhl_data = rna_merge_prepper(vhl_data, gene='VHL', pos_col='hg38_pos', threshold=-3)
print(vhl_data)

In [ ]:
findlay_data=pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna/external_rna_data/20260422_BRCA1_Findlay2018_wRNA.xlsx')
findlay_data = molecular_consequence_mapper(findlay_data,'simplified_consequence')

findlay_data=findlay_data.rename(columns={
                                          'mean.rna.score': 'rna_score'}
                                          )

findlay_data = rna_merge_prepper(findlay_data, gene='BRCA1', pos_col='hg38_start', alt_col='alt_allele', threshold=-2)

print(findlay_data)

In [ ]:
cava_sge = Path('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna')

matches = list(cava_sge.glob("*allscores*"))

all_wrna = [vhl_data, findlay_data]

rna_threshold_dict = {
    'BARD1': -1.244,
    'RAD51D': -2.86,
    'XRCC2': -0.66,
    'PALB2': -1.28709
}
for match in matches:
    gene = str(match).split('/')[-1].split('.')[0]
    gene_df = pd.read_csv(match, sep='\t')

    if len(gene_df.dropna(subset=['RNA_score'])) == 0:
        continue

    gene_df = gene_df.rename(columns={'RNA_score': 'rna_score'})
    gene_df['Gene'] = gene_df['exon'].transform(lambda x: x.split('_')[0])

    rna_threshold = rna_threshold_dict[gene]
    wrna = rna_merge_prepper(gene_df, threshold=rna_threshold)

    all_wrna.append(wrna)

cava_w_rna = pd.concat(all_wrna)
print(cava_w_rna)

df = df.merge(
    cava_w_rna,
    on='pos_id',
    how='left',
    suffixes=('', '_new')
)

if 'maxSpliceAI_new' in df.columns:
    df['maxSpliceAI'] = df['maxSpliceAI'].fillna(df['maxSpliceAI_new'])
    df = df.drop(columns=['maxSpliceAI_new'])

# Fill remaining VHL maxSpliceAI gaps from the Buckley snapshot.
df = df.merge(vhl_spliceai, on='pos_id', how='left', suffixes=('', '_vhl'))
if 'maxSpliceAI_vhl' in df.columns:
    df['maxSpliceAI'] = df['maxSpliceAI'].fillna(df['maxSpliceAI_vhl'])
    df = df.drop(columns=['maxSpliceAI_vhl'])

full_original_df = df

full_original_df.to_excel('./Data/sge_data_for_qc/spliceai_benchmarking/20260822_SGESplicingSet.xlsx', index=False)

# Intial Data Processing and Z-Score Normalization

In [ ]:
normal_mask = df['auth_reported_func_class'] == 'functionally_normal'

df['z_score'] = df.groupby('Gene')['auth_reported_score'].transform(
    lambda x: (x - x[normal_mask.reindex(x.index)].mean()) 
              / x[normal_mask.reindex(x.index)].std()
)

df = df[['Gene', 'auth_reported_score', 'auth_reported_func_class', 'z_score', 
         'simplified_consequence', 'maxSpliceAI', 'splice_impact', 'rna_score', 'rna_consequence']].dropna(subset=['auth_reported_func_class'])

## Z-score Normalization Sanity Check

In [ ]:
z_score_plot = alt.Chart(df).mark_boxplot().encode(
    x='Gene',
    y='z_score:Q'
).facet('auth_reported_func_class')

z_score_plot.display()

# Intron Variants, Z-score normalized

In [ ]:
# Scatter plot helper function

def corr_scatter(df, consequence, rep1, rep2, gene):
    rep1_max = df[rep1].max(axis = 0)
    rep2_max = df[rep2].max(axis = 0)
    rep2_min = df[rep2].min(axis = 0)

    x_max = rep1_max * 1.05
    y_max = rep2_max * 1.05
    y_min = rep2_min * 1.05

    df = df.dropna(subset = [rep1, rep2]).copy()

    df = df.loc[df['simplified_consequence']==consequence]


    scatter = alt.Chart(df).mark_circle().encode(
        x = alt.X(f'{rep1}:Q',
                  scale = alt.Scale(0, x_max)
                  ),
        y = alt.Y(f'{rep2}:Q',
                  scale = alt.Scale(y_min, y_max)
                  ),
        color='splice_impact:N',
        tooltip = ['Gene', 'auth_reported_score']
    )

    corr,_=stats.pearsonr(df[rep1], df[rep2])

    r_text = alt.Chart(pd.DataFrame({
        rep1: [x_max * 0.95],
        rep2: [1],
        'text': [f'r = {corr:.3f}']
    })).mark_text(
        align='right',
        baseline='bottom',
        fontSize=18,
        fontWeight='bold',
        color='black'
    ).encode(
        x = alt.X(f'{rep1}:Q',
                  scale = alt.Scale(0, x_max)),
        y = alt.Y(f'{rep2}:Q',
                  scale = alt.Scale(y_min, y_max)
                  ),
        text='text:N'
    )

    scatter = (scatter+r_text).properties(title = gene).resolve_scale(x = 'shared', y = 'shared').display()

    rep_test = f'{rep1} vs. {rep2}'
    return scatter, gene, rep_test, corr

In [ ]:
def compute_correlations(df, rep1, rep2, consequences=None):
    # Per-gene Pearson r, then a median across genes for the "All (median)" row.

    work = df.dropna(subset=[rep1, rep2]).copy()

    if consequences is None:
        consequences = sorted(work['simplified_consequence'].dropna().unique())
    elif isinstance(consequences, str):
        consequences = [consequences]

    rows = []
    for cons in consequences:
        sub = work.loc[work['simplified_consequence'] == cons]
        if len(sub) < 3:
            continue
        gene_rs = []
        for gene, g in sub.groupby('Gene'):
            if len(g) < 3:
                continue
            if g[rep1].std() == 0 or g[rep2].std() == 0:
                continue  # constant input -> Pearson r is undefined
            r, _ = stats.pearsonr(g[rep1], g[rep2])
            rows.append({'consequence': cons, 'Gene': gene, 'r': r, 'n': len(g)})
            gene_rs.append(r)
        if gene_rs:
            rows.append({
                'consequence': cons,
                'Gene': 'All (median)',
                'r': float(np.median(gene_rs)),
                'n': len(gene_rs)  # number of genes contributing to the median
            })

    return pd.DataFrame(rows)



def corr_heatmap(corr_df, rep1, rep2):
    row_order = list(corr_df['consequence'].unique())
    gene_order = ['All (median)', *corr_df.loc[corr_df['Gene'] != 'All (median)', 'Gene'].unique()]

    base = alt.Chart(corr_df).encode(
        x=alt.X('Gene:N', sort=gene_order),
        y=alt.Y('consequence:N', sort=row_order),
    )

    # Use white text on dark cells (high positive r > 0.5 or strong negative r < -0.6)
    text_color = alt.condition(
        'datum.r > 0.5 || datum.r < -0.6',
        alt.value('white'),
        alt.value('black')
    )

    heatmap = base.mark_rect().encode(
        color=alt.Color('r:Q',
            scale=alt.Scale(scheme='redblue', domainMid=0, domain=[-1, 1]),
            legend=alt.Legend(title='Pearson r')
        ),
        tooltip=['Gene', 'consequence', alt.Tooltip('r:Q', format='.3f'), alt.Tooltip('n:Q', title='n')]
    )

    r_text = base.mark_text(fontSize=12, dy=-5).encode(
        text=alt.Text('r:Q', format='.2f'),
        color=text_color
    )

    # n=variants for per-gene cells; suppressed for the median cell
    n_text = base.mark_text(fontSize=10, dy=7, opacity=0.85).transform_calculate(
        n_label='datum.Gene === "All (median)" ? "" : "n=" + datum.n'
    ).encode(
        text=alt.Text('n_label:N'),
        color=text_color
    )

    return (heatmap + r_text + n_text).properties(title=f'Pearson r: Fitness Score vs. Max SpliceAI', width=500, height = 400)


In [ ]:
scatter, _, _, _ = corr_scatter(df, 'Intron', 'z_score', 'maxSpliceAI', 'All SGE Intron Variants')

In [ ]:
# (label, filtered_df, allowed_consequences or None for all)

consequences = ['Intron', 'Splice Region', 'Missense', 'Synonymous', 'Canonical Splice']

subsets = [
    ('',        df,                                                                  None),
    ('LoF',     df.loc[df['auth_reported_func_class'] == 'functionally_abnormal'],   None),
    ('LoF Low RNA', df.loc[(df['rna_consequence'] == 'low') & (df['auth_reported_func_class']=='functionally_abnormal')],['Missense', 'Synonymous'])
]

corr_df = pd.concat([
    compute_correlations(subset, 'auth_reported_score', 'maxSpliceAI', consequences=[cons])
      .assign(consequence=f'{label} {cons}'.strip())
    for cons in consequences
    for label, subset, allowed_cons in subsets
    if allowed_cons is None or cons in allowed_cons
], ignore_index=True)


# SpliceAI distribution of Normal vars with normal RNA but SpliceAI hit

In [ ]:
normal_splice_hit = full_original_df[(full_original_df['auth_reported_func_class']=='functionally_normal') & (full_original_df['rna_consequence']=='normal') & (full_original_df['simple_splice_impact']=='Splice Impact Predicted')].copy()
normal_splice_hit = normal_splice_hit.dropna(subset='maxSpliceAI')


distribution = alt.Chart(normal_splice_hit[['maxSpliceAI']]).mark_rect().encode(
    x=alt.X('maxSpliceAI:Q', axis=alt.Axis(title='Max SpliceAI'), bin=alt.Bin(maxbins=30)),
    y=alt.Y('count()', axis=alt.Axis(title='Number of Variants')),
).properties(title='SpliceAI for SGE Normal Variants',
             width=650,
             height = 300)

distribution.display()


print(normal_splice_hit)

# Heatmaps a few ways

## Pearson r of Fitness Score vs. Max SpliceAI

In [ ]:
splice_ai_corr_map = corr_heatmap(corr_df, 'z_score', 'maxSpliceAI')

splice_ai_corr_map.display()
splice_ai_corr_map.save(f'/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/pathomechanism/{figure_date_of_record}_FitnessScore_vs_SpliceAI_PearsonR.png', dpi=600)


## % LoF/Normal Variants in SGE vs. Max SpliceAI by Gene

In [ ]:
def compute_spliceai_coverage(df, threshold=0.2, consequences=None):
    work = df.dropna(subset=['maxSpliceAI']).copy()

    if consequences is None:
        consequences = sorted(work['simplified_consequence'].dropna().unique())
    elif isinstance(consequences, str):
        consequences = [consequences]

    rows = []
    for cons in consequences:
        sub = work.loc[work['simplified_consequence'] == cons]
        if len(sub) == 0:
            continue
        for gene, g in sub.groupby('Gene'):
            rows.append({
                'consequence': cons, 'Gene': gene,
                'pct': (g['maxSpliceAI'] >= threshold).mean() * 100,
                'n': len(g)
            })
        rows.append({
            'consequence': cons, 'Gene': 'All',
            'pct': (sub['maxSpliceAI'] >= threshold).mean() * 100,
            'n': len(sub)
        })

    return pd.DataFrame(rows)


def coverage_heatmap(cov_df, threshold, var_type):
    row_order = list(cov_df['consequence'].unique())

    base = alt.Chart(cov_df).encode(
        x=alt.X('Gene:N', sort=['All', *cov_df.loc[cov_df['Gene'] != 'All', 'Gene'].unique()]),
        y=alt.Y('consequence:N', sort=row_order),
    )

    heatmap = base.mark_rect().encode(
        color=alt.Color('pct:Q',
            scale=alt.Scale(scheme='oranges', domain=[0, 100]),
            legend=alt.Legend(title='% predicted')
        ),
        tooltip=['Gene', 'consequence',
                 alt.Tooltip('pct:Q', format='.1f', title='% predicted'),
                 alt.Tooltip('n:Q', title='n')]
    )

    pct_text = base.mark_text(fontSize=12, dy=-5).encode(
        text=alt.Text('pct:Q', format='.1f'),
        color=alt.condition(
            alt.datum.pct > 60, alt.value('white'), alt.value('black')
        )
    )

    n_text = base.mark_text(fontSize=10, dy=7, opacity=0.85).transform_calculate(
        n_label='"n=" + datum.n'
    ).encode(
        text=alt.Text('n_label:N'),
        color=alt.condition(
            alt.datum.pct > 60, alt.value('white'), alt.value('black')
        )
    )

    return (heatmap + pct_text + n_text).properties(
        title=f'% {var_type} variants with maxSpliceAI ≥ {threshold}', width=500, height = 300
    )


In [ ]:
consequences = ['Intron', 'Splice Region', 'Missense', 'Synonymous', 'Canonical Splice']

subsets = [
    ('LoF',     df.loc[df['auth_reported_func_class'] == 'functionally_abnormal'], None),
    ('Low RNA LoF', df.loc[(df['rna_consequence'] == 'low') & (df['auth_reported_func_class'] == 'functionally_abnormal')],['Missense', 'Synonymous', 'Splice Region']),
]

cov_df = pd.concat([
    compute_spliceai_coverage(subset, threshold=0.2, consequences=[cons])
      .assign(consequence=f'{label} {cons}'.strip())
    for cons in consequences
    for label, subset, allowed_cons in subsets
    if allowed_cons is None or cons in allowed_cons
], ignore_index=True)

lof_spliceai_map = coverage_heatmap(cov_df, threshold=0.2, var_type = 'LoF')

lof_spliceai_map.display()
lof_spliceai_map.save(f'/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/pathomechanism/{figure_date_of_record}_ProportionLOF_wSpliceAIHit.png', dpi=1200)


In [ ]:
consequences = ['Intron', 'Splice Region', 'Missense', 'Synonymous', 'Canonical Splice']

subsets = [
    ('Normal',            df.loc[df['auth_reported_func_class'] == 'functionally_normal'], None),
    ('Normal RNA Normal', df.loc[(df['rna_consequence'] == 'normal') & (df['auth_reported_func_class'] == 'functionally_normal')], ['Missense', 'Synonymous', 'Splice Region']),
]

cov_df = pd.concat([
    compute_spliceai_coverage(subset, threshold=0.2, consequences=[cons])
      .assign(consequence=f'{label} {cons}'.strip())
    for cons in consequences
    for label, subset, allowed_cons in subsets
    if allowed_cons is None or cons in allowed_cons
], ignore_index=True)

normal_splice_ai_map = coverage_heatmap(cov_df, threshold=0.2, var_type='Normal')

normal_splice_ai_map.display()
normal_splice_ai_map.save(f'/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/pathomechanism/{figure_date_of_record}_ProportionNormal_wSpliceAIHit.png', dpi=600)

# Balanced Accuracy

In [ ]:
def compute_balanced_accuracy(df, threshold=0.2, label_col='auth_reported_func_class',
                               pos_label='functionally_abnormal', neg_label='functionally_normal',
                               consequences=None):
    work = df.dropna(subset=['maxSpliceAI']).copy()
    work = work[work[label_col].isin([pos_label, neg_label])]

    if consequences is None:
        consequences = sorted(work['simplified_consequence'].dropna().unique())
    elif isinstance(consequences, str):
        consequences = [consequences]

    rows = []
    for cons in consequences:
        sub = work.loc[work['simplified_consequence'] == cons]
        if len(sub) == 0:
            continue
        for gene, g in sub.groupby('Gene'):
            pos = g[g[label_col] == pos_label]
            neg = g[g[label_col] == neg_label]
            if len(pos) == 0 or len(neg) == 0:
                continue
            sens = (pos['maxSpliceAI'] >= threshold).mean()
            spec = (neg['maxSpliceAI'] < threshold).mean()
            rows.append({
                'consequence': cons, 'Gene': gene,
                'sensitivity': sens, 'specificity': spec,
                'balanced_acc': (sens + spec) / 2,
                'n_pos': len(pos), 'n_neg': len(neg),
            })
        pos_all = sub[sub[label_col] == pos_label]
        neg_all = sub[sub[label_col] == neg_label]
        if len(pos_all) > 0 and len(neg_all) > 0:
            sens = (pos_all['maxSpliceAI'] >= threshold).mean()
            spec = (neg_all['maxSpliceAI'] < threshold).mean()
            rows.append({
                'consequence': cons, 'Gene': 'All',
                'sensitivity': sens, 'specificity': spec,
                'balanced_acc': (sens + spec) / 2,
                'n_pos': len(pos_all), 'n_neg': len(neg_all),
            })

    return pd.DataFrame(rows)


def balanced_accuracy_heatmap(ba_df, threshold, var_type):
    row_order = list(ba_df['consequence'].unique())

    base = alt.Chart(ba_df).encode(
        x=alt.X('Gene:N', sort=['All', *ba_df.loc[ba_df['Gene'] != 'All', 'Gene'].unique()]),
        y=alt.Y('consequence:N', sort=row_order),
    )

    text_color = alt.condition(
        'datum.balanced_acc > 0.75 || datum.balanced_acc < 0.35',
        alt.value('white'), alt.value('black')
    )

    heatmap = base.mark_rect().encode(
        color=alt.Color('balanced_acc:Q',
            scale=alt.Scale(scheme='redblue', domainMid=0.5, domain=[0, 1]),
            legend=alt.Legend(title='Balanced Accuracy')
        ),
        tooltip=['Gene', 'consequence',
                 alt.Tooltip('balanced_acc:Q', format='.3f', title='Balanced Accuracy'),
                 alt.Tooltip('n_pos:Q', title='n abnormal'),
                 alt.Tooltip('n_neg:Q', title='n normal')]
    )

    ba_text = base.mark_text(fontSize=12, dy=-5).encode(
        text=alt.Text('balanced_acc:Q', format='.2f'), color=text_color
    )
    n_text = base.mark_text(fontSize=10, dy=7, opacity=0.85).transform_calculate(
        n_label='"+" + datum.n_pos + "/-" + datum.n_neg'
    ).encode(text=alt.Text('n_label:N'), color=text_color)

    return (heatmap + ba_text + n_text).properties(
        title=f'Balanced Accuracy: {var_type} (maxSpliceAI ≥ {threshold})', width=650, height=300
    )

In [ ]:
consequences_standard = ['Intron', 'Splice Region']
consequences_rna      = ['Missense', 'Synonymous']

# For missense/synonymous: positive class = functionally_abnormal AND low RNA only
rna_lof_df = df[
    (df['auth_reported_func_class'] == 'functionally_normal') |
    ((df['auth_reported_func_class'] == 'functionally_abnormal') & (df['rna_consequence'] == 'low'))
].copy()

ba_df = pd.concat(
    [compute_balanced_accuracy(df,         threshold=0.2, consequences=[cons]).assign(consequence=cons)
     for cons in consequences_standard] +
    [compute_balanced_accuracy(rna_lof_df, threshold=0.2, consequences=[cons]).assign(consequence=cons)
     for cons in consequences_rna],
    ignore_index=True
)

ba_map = balanced_accuracy_heatmap(ba_df, threshold=0.2, var_type='All Variants')
ba_map.display()

# Specificity/Sensitivity Heatmaps

In [ ]:
def sensitivity_heatmap(ba_df, threshold, var_type):
    row_order = list(ba_df['consequence'].unique())

    base = alt.Chart(ba_df).encode(
        x=alt.X('Gene:N', sort=['All', *ba_df.loc[ba_df['Gene'] != 'All', 'Gene'].unique()]),
        y=alt.Y('consequence:N', sort=row_order),
    )

    text_color = alt.condition(
        'datum.sensitivity > 0.7', alt.value('white'), alt.value('black')
    )

    heatmap = base.mark_rect().encode(
        color=alt.Color('sensitivity:Q',
            scale=alt.Scale(scheme='blues', domain=[0, 1]),
            legend=alt.Legend(title='Sensitivity')
        ),
        tooltip=['Gene', 'consequence',
                 alt.Tooltip('sensitivity:Q', format='.3f', title='Sensitivity'),
                 alt.Tooltip('n_pos:Q', title='n abnormal')]
    )

    val_text = base.mark_text(fontSize=12, dy=-5).encode(
        text=alt.Text('sensitivity:Q', format='.2f'), color=text_color
    )
    n_text = base.mark_text(fontSize=10, dy=7, opacity=0.85).transform_calculate(
        n_label='"n=" + datum.n_pos'
    ).encode(text=alt.Text('n_label:N'), color=text_color)

    return (heatmap + val_text + n_text).properties(
        title=f'Sensitivity: {var_type} (maxSpliceAI ≥ {threshold})', width=500, height=300
    )


def specificity_heatmap(ba_df, threshold, var_type):
    row_order = list(ba_df['consequence'].unique())

    base = alt.Chart(ba_df).encode(
        x=alt.X('Gene:N', sort=['All', *ba_df.loc[ba_df['Gene'] != 'All', 'Gene'].unique()]),
        y=alt.Y('consequence:N', sort=row_order),
    )

    text_color = alt.condition(
        'datum.specificity > 0.7', alt.value('white'), alt.value('black')
    )

    heatmap = base.mark_rect().encode(
        color=alt.Color('specificity:Q',
            scale=alt.Scale(scheme='greens', domain=[0, 1]),
            legend=alt.Legend(title='Specificity')
        ),
        tooltip=['Gene', 'consequence',
                 alt.Tooltip('specificity:Q', format='.3f', title='Specificity'),
                 alt.Tooltip('n_neg:Q', title='n normal')]
    )

    val_text = base.mark_text(fontSize=12, dy=-5).encode(
        text=alt.Text('specificity:Q', format='.2f'), color=text_color
    )
    n_text = base.mark_text(fontSize=10, dy=7, opacity=0.85).transform_calculate(
        n_label='"n=" + datum.n_neg'
    ).encode(text=alt.Text('n_label:N'), color=text_color)

    return (heatmap + val_text + n_text).properties(
        title=f'Specificity: {var_type} (maxSpliceAI ≥ {threshold})', width=500, height=300
    )

In [ ]:
sens_map = sensitivity_heatmap(ba_df, threshold=0.2, var_type='All Variants')
spec_map = specificity_heatmap(ba_df, threshold=0.2, var_type='All Variants')

sens_map.display()
spec_map.display()

# Base Change Distirbution Analysis
## Overall LoF Intronic Variant Distribution

In [ ]:
intron_df = cleaned_raw_df[(cleaned_raw_df['simplified_consequence'] == 'Intron') & (cleaned_raw_df['auth_reported_func_class'].isin(['functionally_abnormal', 'functionally_normal']))]
intron_df['simple_splice_impact'] = 'Splice Impact Predicted'
intron_df.loc[intron_df['splice_impact'] == '< Threshold', 'simple_splice_impact'] = '< Threshold'

intron_df = intron_df.dropna(subset=['hgvs_c'])
intron_df['base_change'] = intron_df['hgvs_c'].transform(lambda x: x[-3:])
intron_df['intronic_dist'] = intron_df['hgvs_c'].str.extract(r'\d[+-](\d+)').astype(int)
intron_df = intron_df[(intron_df['intronic_dist'] < 50) & (intron_df['intronic_dist'] >= 9)].copy()
                                                                                  
lof_intron_df = intron_df[intron_df['auth_reported_func_class']=='functionally_abnormal']

In [ ]:
plp_intron_summary_df = pd.concat([lof_intron_df.value_counts(subset='base_change'), lof_intron_df.value_counts(subset='base_change', normalize=True)], axis=1, keys=['count', 'proportion']).reset_index()
print(plp_intron_summary_df)
plp_base_proportions = alt.Chart(plp_intron_summary_df).mark_bar().encode(
    x = 'base_change:N',
    y='proportion:Q'
).properties(title = 'Proportion of Base Changes in Near Intronic PLPs')

plp_base_proportions.display()

## SpliceAI Hit Rate

In [ ]:
grouped = intron_df.groupby(['base_change', 'auth_reported_func_class'])

processed_tuples = []

def get_counts(summary, label):
    """Safely extract count + proportion for a given simple_splice_impact label."""
    row = summary[summary['simple_splice_impact'] == label]
    if row.empty:
        return 0, 0.0
    return row['count'].iloc[0], row['proportion'].iloc[0]

processed_tuples = []

for (base_change, func_class), df in grouped:

    summary = pd.concat([
        df.value_counts(subset='simple_splice_impact'),
        df.value_counts(subset='simple_splice_impact', normalize=True)
    ], axis=1, keys=['count', 'proportion']).reset_index()

    if func_class == 'functionally_normal':
        correct_count, correct_proportion = get_counts(summary, '< Threshold')
        incorrect_count, incorrect_proportion = get_counts(summary, 'Splice Impact Predicted')
    else:
        correct_count, correct_proportion = get_counts(summary, 'Splice Impact Predicted')
        incorrect_count, incorrect_proportion = get_counts(summary, '< Threshold')

    processed_tuples.append((func_class, base_change, correct_count, incorrect_count, correct_proportion, incorrect_proportion))

base_change_summary_df = pd.DataFrame(processed_tuples, columns=[
    'clinical_significance', 'base_change', 'correct_count', 'incorrect_count', 'correct_proportion', 'incorrect_proportion'
]).sort_values(['clinical_significance', 'correct_proportion'], ascending=False).reset_index(drop=True)

print(base_change_summary_df)

In [ ]:
base_change_heatmap = alt.Chart(base_change_summary_df).mark_rect().encode(
    x=alt.X('clinical_significance:N'),
    y=alt.Y('base_change:N'),
    color=alt.Color('incorrect_proportion:Q'),
    tooltip = 'incorrect_proportion:Q'
)

base_change_heatmap.display()

# Summary Bars

In [ ]:
wRNA_df = full_original_df[full_original_df['Gene'].isin(['BARD1', 'RAD51D', 'XRCC2', 'BRCA1', 'VHL'])].copy()

wRNA_mis_syn = wRNA_df[wRNA_df['simplified_consequence'].isin(['Missense', 'Synonymous'])]

In [ ]:
def build_summary_tuples(df, group_col, classifiers):
    tuples = []
    for consequence, group in df.groupby(group_col):
        total = len(group)
        for name, condition_fn in classifiers:
            mask = condition_fn(group)
            tuples.append((name, consequence, int(mask.sum()), total, mask.mean()))
    return tuples

In [ ]:
# True  → SpliceAI: proportion that are (low RNA for mis/syn | LoF for intron/splice) AND flagged by SpliceAI
# False → SpliceAI: proportion flagged by SpliceAI regardless of RNA consequence / functional class
FILTERED = False

In [ ]:
func_abnormal = wRNA_mis_syn[wRNA_mis_syn['auth_reported_func_class'] == 'functionally_abnormal']

if FILTERED:
    spliceai_cond = lambda df: (df['simple_splice_impact'] == 'Splice Impact Predicted') & (df['rna_consequence'] == 'low')
    analysis_type='correct_bar'
else:
    spliceai_cond = lambda df: df['simple_splice_impact'] == 'Splice Impact Predicted'
    analysis_type='predictbar'
summary_tuples = build_summary_tuples(
    func_abnormal,
    group_col='simplified_consequence',
    classifiers=[
        ('SGE',      lambda df: df['rna_consequence'] == 'low'),
        ('SpliceAI', spliceai_cond),
    ]
)

In [ ]:
all_splice_defect = full_original_df[full_original_df['simplified_consequence'].isin(['Intron', 'Splice Region'])]

if FILTERED:
    spliceai_cond = lambda df: (df['simple_splice_impact'] == 'Splice Impact Predicted') & (df['auth_reported_func_class'] == 'functionally_abnormal')
else:
    spliceai_cond = lambda df: df['simple_splice_impact'] == 'Splice Impact Predicted'

summary_tuples += build_summary_tuples(
    all_splice_defect,
    group_col='simplified_consequence',
    classifiers=[
        ('SGE',      lambda df: df['auth_reported_func_class'] == 'functionally_abnormal'),
        ('SpliceAI', spliceai_cond),
    ]
)

In [ ]:
proportion_df = pd.DataFrame(summary_tuples, columns=['classifier', 'Consequence', 'count', 'total', 'proportion'])

print(proportion_df)

In [ ]:
@alt.theme.register('arial', enable=True)
def arial_theme():
    return {
        'config': {
            'axis':   {'labelFont': 'Arial', 'titleFont': 'Arial'},
            'legend': {'labelFont': 'Arial', 'titleFont': 'Arial'},
            'header': {'labelFont': 'Arial', 'titleFont': 'Arial'},
        }
    }

def bar_helper(df, spacing=80):
    return alt.Chart(df).mark_bar().encode(
        x=alt.X('classifier:N', axis=alt.Axis(labelAngle=0)),
        y=alt.Y('proportion:Q'),
        color='classifier:N'
    ).properties(width = 175, height = 200).facet('Consequence:N', spacing=spacing)

In [ ]:
mis_syn_bar = bar_helper(proportion_df[proportion_df['Consequence'].isin(['Missense', 'Synonymous'])])
intron_splice_bar = bar_helper(proportion_df[proportion_df['Consequence'].isin(['Intron', 'Splice Region'])])

splice_ai_correct_bars = alt.hconcat(mis_syn_bar, intron_splice_bar).configure_view(stroke = None).configure_axis(grid = False)

splice_ai_correct_bars.display()

splice_ai_correct_bars.save(f'/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/pathomechanism/{figure_date_of_record}_SpliceAI_{analysis_type}.svg')

In [ ]:
def two_prop_ztest(count1, n1, count2, n2):
    p1, p2 = count1 / n1, count2 / n2
    p_pool = (count1 + count2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    z = (p1 - p2) / se
    p_val = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p_val

rows = []
for consequence, group in proportion_df.groupby('Consequence'):
    sge      = group[group['classifier'] == 'SGE'].iloc[0]
    spliceai = group[group['classifier'] == 'SpliceAI'].iloc[0]
    z, p_val = two_prop_ztest(sge['count'], sge['total'], spliceai['count'], spliceai['total'])
    rows.append({
        'Consequence':        consequence,
        'SGE_proportion':     sge['proportion'],
        'SpliceAI_proportion': spliceai['proportion'],
        'z_stat':             z,
        'p_value':            p_val,
    })

ztest_df = pd.DataFrame(rows)
print(ztest_df)

# PR-AUC Summary

In [ ]:
#Get AUC helperfunction
def get_auc(
    df: pd.DataFrame,
    score_col: str = "maxSpliceAI",
    label_col: str = "ClinicalSignificance",
    pos_label: str = "PLP",
    neg_label: str = "BLB"):

    df = df.dropna(subset=score_col)

    sub = df[df[label_col].isin([pos_label, neg_label])].copy()
    
    if len(sub) < 10:
        print('Sample size too small...')
        return None

    y_true = (sub[label_col] == pos_label).astype(int)
    y_score = sub[score_col]
    print(sub.loc[sub[score_col].isnull()])

    pr_auc=roc_auc_score(y_true, y_score)

    '''
    precision, recall, _ = precision_recall_curve(y_true, y_score)

    pr_auc = auc(recall, precision)
    '''
    return pr_auc

In [ ]:
consequences = full_original_df['simplified_consequence'].unique().tolist()

auc_tuples = []
for consequence in consequences:

    skips = ['Stop Lost', 'Start Lost', 'UTR Variant', 'Stop Gained']
    presumed_rna=['Intron', 'Splice Region', 'Canonical Splice']

    measured_rna=['Missense', 'Synonymous']

    if consequence in skips:
        continue

    elif consequence in presumed_rna:
        df = full_original_df[full_original_df['simplified_consequence']==consequence]
    elif consequence in measured_rna:
        df = full_original_df[(full_original_df['simplified_consequence']==consequence) & (full_original_df['rna_consequence']=='low')]


    consequence_auc = get_auc(df, label_col='auth_reported_func_class', pos_label='functionally_abnormal', neg_label='functionally_normal')
    
    auc_tuples.append((consequence, consequence_auc))


auc_df = pd.DataFrame(auc_tuples, columns=['consequence', 'auc'])

auc_df.head()

In [ ]:
auc_bars = alt.Chart(auc_df).mark_bar().encode(
    x=alt.X('consequence:N'),
    y=alt.Y('auc:Q'),
    color=alt.Color('consequence:N')
)

auc_bars.display()